# TLS/QUIC Malware Detection — Model Training

## Objective
Train an XGBoost classifier to detect encrypted malware traffic:
- **Encrypted C2 (Command & Control)** via TLS/QUIC channels
- **Beaconing** — periodic callbacks to C2 infrastructure
- **Malicious tunneling** — covert data exfil over DNS-over-HTTPS / QUIC

## Datasets
- **CIC-MalAnal2020** — malware network traffic captures with CICFlowMeter features
- **Brad Duncan Malware Traffic Analysis** — real-world malware PCAPs with flow-level labels

## Pipeline
1. Load & merge labeled flow records from both datasets
2. Handle class imbalance (benign vs malware)
3. Train optimized XGBoost with hyperparameter tuning
4. Threshold tuning via ROC-AUC & Precision-Recall curves
5. Export model artifact as `.pkl` for streaming inference

---
## 1. Setup & Imports

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Add project root to path
PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"XGBoost version: {xgb.__version__}")

---
## 2. Configuration

In [ ]:
# === Paths ===
DATA_DIR = PROJECT_ROOT / "data_and_demo"
MODEL_DIR = Path().resolve() / "models"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_ARTIFACT_PATH = MODEL_DIR / "malware_tls_streaming_model.pkl"

# === Training config ===
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # From training set

# === Feature list (must match parser.py ALL_FEATURES) ===
# These are loaded from the parser module
from ml_engine.malware_tls.parser import ALL_FEATURES, FEATURE_COUNT
print(f"Expected feature count: {FEATURE_COUNT}")

---
## 3. Data Loading & Merging

Load labeled flow records from CIC-MalAnal2020 and Brad Duncan datasets.

### Expected Data Formats

**CIC-MalAnal2020 CSV** (output of CICFlowMeter):
- Columns: `Src IP`, `Dst IP`, `Src Port`, `Dst Port`, `Protocol`, `Flow Duration`, ...
- Label column: `Label` (BENIGN / Malware)

**Brad Duncan CSV** (custom format):
- Columns: `src_ip`, `dst_ip`, `src_port`, `dst_port`, `protocol`, `fwd_pkts`, `bwd_pkts`, ...
- Label column: `label` (benign / malware)

In [ ]:
from ml_engine.malware_tls.parser import csv_to_flow_dataframe

# === Load datasets ===
# Adjust these paths to your actual dataset locations
dataset_paths = [
    # CIC-MalAnal2020 datasets
    DATA_DIR / "CIC-MalAnal2020" / "Wednesday-WorkingHours.pcap_ISCX.csv",
    DATA_DIR / "CIC-MalAnal2020" / "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    DATA_DIR / "CIC-MalAnal2020" / "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    # Brad Duncan datasets
    DATA_DIR / "brad_duncan" / "malware_traffic.csv",
]

dataframes = []
for path in dataset_paths:
    if path.exists():
        print(f"Loading: {path.name}")
        df = csv_to_flow_dataframe(str(path))
        print(f"  → {len(df)} flows, columns: {list(df.columns[:5])}...")
        dataframes.append(df)
    else:
        print(f"  ⚠ Not found: {path}")

if not dataframes:
    print("\n⚠ No datasets found. Generating synthetic data for demonstration.")
    print("  Place your CSV files in data_and_demo/ and re-run this cell.")
    USE_SYNTHETIC = True
else:
    USE_SYNTHETIC = False
    df_all = pd.concat(dataframes, ignore_index=True)
    print(f"\nCombined dataset: {len(df_all)} flows")

In [ ]:
# === Synthetic data generator (for demonstration) ===
def generate_synthetic_dataset(n_samples: int = 10000, malware_ratio: float = 0.15):
    """
    Generate synthetic flow-level features mimicking real TLS/QUIC malware traffic.
    Malware flows have distinct patterns:
    - Regular beaconing (low IAT CV)
    - Deprecated cipher suites
    - Self-signed certificates
    - No SNI or suspicious SNI
    - Small, periodic data transfers
    """
    np.random.seed(RANDOM_STATE)
    n_malware = int(n_samples * malware_ratio)
    n_benign = n_samples - n_malware
    
    rows = []
    
    # --- Generate benign traffic ---
    for _ in range(n_benign):
        row = {f: 0.0 for f in ALL_FEATURES}
        # Normal flow metrics
        row['src_port'] = np.random.randint(49152, 65535)
        row['dst_port'] = np.random.choice([80, 443, 8080, 8443, 53])
        row['fwd_pkt_count'] = np.random.randint(5, 500)
        row['bwd_pkt_count'] = np.random.randint(3, 400)
        row['total_pkt_count'] = row['fwd_pkt_count'] + row['bwd_pkt_count']
        row['fwd_byte_count'] = np.random.randint(500, 500000)
        row['bwd_byte_count'] = np.random.randint(500, 2000000)
        row['total_byte_count'] = row['fwd_byte_count'] + row['bwd_byte_count']
        row['fwd_ttl_mean'] = np.random.choice([64, 128, 255])
        row['bwd_ttl_mean'] = np.random.choice([48, 64, 128])
        row['flow_duration_sec'] = np.random.exponential(5.0)
        # Normal IAT (high variance)
        row['fwd_iat_mean'] = np.random.exponential(0.5)
        row['fwd_iat_std'] = row['fwd_iat_mean'] * np.random.uniform(0.5, 2.0)
        row['bwd_iat_mean'] = np.random.exponential(0.3)
        row['bwd_iat_std'] = row['bwd_iat_mean'] * np.random.uniform(0.5, 2.0)
        row['overall_iat_mean'] = np.random.exponential(0.2)
        row['overall_iat_std'] = row['overall_iat_mean'] * np.random.uniform(0.5, 2.5)
        row['pkt_size_mean'] = np.random.uniform(200, 1400)
        row['pkt_size_std'] = np.random.uniform(100, 600)
        row['fwd_pkt_size_mean'] = np.random.uniform(100, 1400)
        row['bwd_pkt_size_mean'] = np.random.uniform(200, 1400)
        # Normal TLS: TLS 1.2/1.3, many ciphers, valid SNI
        row['tls_1_2'] = 1.0 if np.random.random() > 0.3 else 0.0
        row['tls_1_3'] = 1.0 if np.random.random() > 0.5 else 0.0
        row['cipher_suite_count'] = np.random.randint(10, 30)
        row['has_deprecated_cipher'] = 0.0
        row['has_sni'] = 1.0
        row['sni_length'] = np.random.randint(5, 50)
        row['sni_entropy'] = np.random.uniform(2.0, 4.0)
        row['cert_self_signed'] = 0.0
        row['cert_expired'] = 0.0
        row['cert_valid_days_remaining'] = np.random.randint(30, 400)
        row['ja4_present'] = 1.0 if np.random.random() > 0.2 else 0.0
        row['ja3_present'] = 1.0 if np.random.random() > 0.3 else 0.0
        row['tls_extension_count'] = np.random.randint(8, 20)
        row['has_alpn'] = 1.0
        row['alpn_h2'] = 1.0 if np.random.random() > 0.4 else 0.0
        row['alpn_http1_1'] = 1.0 if np.random.random() > 0.6 else 0.0
        # Beaconing: high CV = irregular
        row['iat_cv'] = np.random.uniform(0.8, 3.0)
        row['iat_regularity_score'] = np.random.uniform(0.0, 0.4)
        row['dominant_freq_hz'] = np.random.uniform(0.0, 0.01)
        row['dominant_freq_power'] = np.random.uniform(0.0, 0.5)
        row['consecutive_similar_iat_count'] = np.random.randint(0, 3)
        # Flags
        row['syn_count'] = np.random.randint(1, 5)
        row['ack_count'] = int(row['total_pkt_count'] * 0.7)
        row['fin_count'] = np.random.randint(0, 3)
        row['rst_count'] = 0.0
        row['psh_count'] = np.random.randint(0, 10)
        # Ratios
        row['fwd_bwd_pkt_ratio'] = row['fwd_pkt_count'] / max(row['bwd_pkt_count'], 1)
        row['fwd_bwd_byte_ratio'] = row['fwd_byte_count'] / max(row['bwd_byte_count'], 1)
        row['ack_ratio'] = row['ack_count'] / max(row['total_pkt_count'], 1)
        # Protocol
        row['proto_tcp'] = 1.0
        row['proto_udp'] = 0.0
        row['proto_icmp'] = 0.0
        row['proto_quic'] = 0.0
        row['label'] = 'benign'
        rows.append(row)
    
    # --- Generate malware traffic ---
    malware_types = ['c2_beacon', 'c2_tls', 'dns_tunnel', 'quic_malware']
    for _ in range(n_malware):
        mtype = np.random.choice(malware_types)
        row = {f: 0.0 for f in ALL_FEATURES}
        
        if mtype == 'c2_beacon':
            # Periodic C2 beaconing over TLS
            row['src_port'] = np.random.randint(49152, 65535)
            row['dst_port'] = 443
            row['fwd_pkt_count'] = np.random.randint(2, 15)
            row['bwd_pkt_count'] = np.random.randint(1, 10)
            row['total_pkt_count'] = row['fwd_pkt_count'] + row['bwd_pkt_count']
            row['fwd_byte_count'] = np.random.randint(100, 5000)
            row['bwd_byte_count'] = np.random.randint(200, 10000)
            row['total_byte_count'] = row['fwd_byte_count'] + row['bwd_byte_count']
            row['flow_duration_sec'] = np.random.uniform(0.5, 5.0)
            row['fwd_ttl_mean'] = np.random.choice([64, 128])
            row['bwd_ttl_mean'] = np.random.randint(40, 128)
            # Regular beaconing: LOW IAT CV
            row['iat_cv'] = np.random.uniform(0.02, 0.15)
            row['iat_regularity_score'] = np.random.uniform(0.8, 1.0)
            row['dominant_freq_hz'] = np.random.uniform(0.001, 0.05)
            row['dominant_freq_power'] = np.random.uniform(2.0, 8.0)
            row['consecutive_similar_iat_count'] = np.random.randint(10, 60)
            row['fwd_iat_mean'] = np.random.uniform(5.0, 120.0)
            row['fwd_iat_std'] = row['fwd_iat_mean'] * np.random.uniform(0.02, 0.1)
            row['overall_iat_mean'] = row['fwd_iat_mean'] * 0.5
            row['overall_iat_std'] = row['overall_iat_mean'] * np.random.uniform(0.03, 0.12)
            row['pkt_size_mean'] = np.random.uniform(100, 400)
            row['pkt_size_std'] = np.random.uniform(10, 50)
            # TLS anomalies
            row['tls_1_2'] = 1.0
            row['cipher_suite_count'] = np.random.randint(1, 6)
            row['has_deprecated_cipher'] = 1.0 if np.random.random() > 0.5 else 0.0
            row['has_sni'] = 0.0 if np.random.random() > 0.6 else 1.0
            row['cert_self_signed'] = 1.0 if np.random.random() > 0.4 else 0.0
            row['cert_expired'] = 1.0 if np.random.random() > 0.8 else 0.0
            row['cert_valid_days_remaining'] = np.random.randint(-30, 365)
            row['sni_length'] = np.random.randint(0, 30)
            row['sni_entropy'] = np.random.uniform(3.0, 5.0)
            row['proto_tcp'] = 1.0
        
        elif mtype == 'c2_tls':
            # TLS-based C2 with deprecated ciphers
            row['src_port'] = np.random.randint(49152, 65535)
            row['dst_port'] = np.random.choice([443, 8443, 4443, 9443])
            row['fwd_pkt_count'] = np.random.randint(10, 100)
            row['bwd_pkt_count'] = np.random.randint(5, 80)
            row['total_pkt_count'] = row['fwd_pkt_count'] + row['bwd_pkt_count']
            row['fwd_byte_count'] = np.random.randint(1000, 50000)
            row['bwd_byte_count'] = np.random.randint(5000, 200000)
            row['total_byte_count'] = row['fwd_byte_count'] + row['bwd_byte_count']
            row['flow_duration_sec'] = np.random.uniform(10, 300)
            row['iat_cv'] = np.random.uniform(0.3, 1.5)
            row['iat_regularity_score'] = np.random.uniform(0.2, 0.7)
            row['consecutive_similar_iat_count'] = np.random.randint(2, 15)
            row['tls_1_0'] = 1.0 if np.random.random() > 0.7 else 0.0
            row['tls_1_2'] = 1.0
            row['cipher_suite_count'] = np.random.randint(1, 5)
            row['has_deprecated_cipher'] = 1.0
            row['has_sni'] = 1.0
            row['sni_length'] = np.random.randint(10, 40)
            row['sni_entropy'] = np.random.uniform(3.5, 5.0)
            row['cert_self_signed'] = 1.0
            row['cert_valid_days_remaining'] = np.random.randint(0, 30)
            row['proto_tcp'] = 1.0
        
        elif mtype == 'dns_tunnel':
            # DNS-over-HTTPS / DNS tunneling
            row['src_port'] = np.random.randint(49152, 65535)
            row['dst_port'] = np.random.choice([443, 853, 53])
            row['fwd_pkt_count'] = np.random.randint(50, 500)
            row['bwd_pkt_count'] = np.random.randint(50, 500)
            row['total_pkt_count'] = row['fwd_pkt_count'] + row['bwd_pkt_count']
            row['fwd_byte_count'] = np.random.randint(5000, 500000)
            row['bwd_byte_count'] = np.random.randint(5000, 2000000)
            row['total_byte_count'] = row['fwd_byte_count'] + row['bwd_byte_count']
            row['flow_duration_sec'] = np.random.uniform(30, 600)
            row['fwd_pkt_size_mean'] = np.random.uniform(50, 200)
            row['bwd_pkt_size_mean'] = np.random.uniform(200, 1400)
            row['iat_cv'] = np.random.uniform(0.5, 2.0)
            row['tls_1_2'] = 1.0
            row['cipher_suite_count'] = np.random.randint(5, 15)
            row['has_sni'] = 1.0
            row['sni_length'] = np.random.randint(20, 60)
            row['sni_entropy'] = np.random.uniform(4.0, 5.0)
            row['proto_udp'] = 1.0
            row['proto_tcp'] = 0.0
        
        elif mtype == 'quic_malware':
            # QUIC-based malware
            row['src_port'] = np.random.randint(49152, 65535)
            row['dst_port'] = 443
            row['fwd_pkt_count'] = np.random.randint(5, 50)
            row['bwd_pkt_count'] = np.random.randint(3, 30)
            row['total_pkt_count'] = row['fwd_pkt_count'] + row['bwd_pkt_count']
            row['fwd_byte_count'] = np.random.randint(500, 20000)
            row['bwd_byte_count'] = np.random.randint(1000, 50000)
            row['total_byte_count'] = row['fwd_byte_count'] + row['bwd_byte_count']
            row['flow_duration_sec'] = np.random.uniform(1, 30)
            row['tls_quic'] = 1.0
            row['proto_quic'] = 1.0
            row['proto_tcp'] = 0.0
            row['proto_udp'] = 1.0
            row['iat_cv'] = np.random.uniform(0.05, 0.3)
            row['iat_regularity_score'] = np.random.uniform(0.6, 1.0)
            row['consecutive_similar_iat_count'] = np.random.randint(5, 30)
            row['cipher_suite_count'] = np.random.randint(1, 8)
            row['has_sni'] = 0.0 if np.random.random() > 0.5 else 1.0
            row['cert_self_signed'] = 1.0 if np.random.random() > 0.3 else 0.0
        
        # Common malware indicators
        row['fwd_ttl_mean'] = np.random.choice([64, 128])
        row['bwd_ttl_mean'] = np.random.randint(40, 128)
        row['fwd_ttl_std'] = np.random.uniform(0, 10)
        row['bwd_ttl_std'] = np.random.uniform(0, 10)
        row['fwd_iat_max'] = row['fwd_iat_mean'] * np.random.uniform(1.5, 5.0)
        row['fwd_iat_min'] = row['fwd_iat_mean'] * np.random.uniform(0.1, 0.8)
        row['bwd_iat_mean'] = np.random.exponential(0.5)
        row['bwd_iat_std'] = row['bwd_iat_mean'] * np.random.uniform(0.3, 1.5)
        row['bwd_iat_max'] = row['bwd_iat_mean'] * np.random.uniform(1.5, 5.0)
        row['bwd_iat_min'] = row['bwd_iat_mean'] * np.random.uniform(0.1, 0.8)
        row['fwd_pkt_size_max'] = row['fwd_pkt_size_mean'] * np.random.uniform(1.2, 3.0)
        row['bwd_pkt_size_max'] = row['bwd_pkt_size_mean'] * np.random.uniform(1.2, 3.0)
        row['syn_count'] = np.random.randint(1, 3)
        row['ack_count'] = int(row['total_pkt_count'] * 0.8)
        row['fin_count'] = np.random.randint(0, 2)
        row['rst_count'] = 0.0
        row['psh_count'] = np.random.randint(0, 5)
        row['fwd_bwd_pkt_ratio'] = row['fwd_pkt_count'] / max(row['bwd_pkt_count'], 1)
        row['fwd_bwd_byte_ratio'] = row['fwd_byte_count'] / max(row['bwd_byte_count'], 1)
        row['ack_ratio'] = row['ack_count'] / max(row['total_pkt_count'], 1)
        row['label'] = 'malware'
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)


if USE_SYNTHETIC:
    df_all = generate_synthetic_dataset(n_samples=10000, malware_ratio=0.15)
    print(f"Generated synthetic dataset: {len(df_all)} flows")
    print(f"  Benign: {(df_all['label'] == 'benign').sum()}")
    print(f"  Malware: {(df_all['label'] == 'malware').sum()}")

---
## 4. Data Exploration & Label Distribution

In [ ]:
# === Label distribution ===
print("Dataset shape:", df_all.shape)
print("\nLabel distribution:")
label_counts = df_all['label'].value_counts()
print(label_counts)
print(f"\nClass imbalance ratio: {label_counts.min() / label_counts.max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
label_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Pie chart
label_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[1].set_title('Class Proportions')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# === Feature distributions: benign vs malware ===
key_features = [
    'iat_cv', 'iat_regularity_score', 'cipher_suite_count',
    'has_deprecated_cipher', 'cert_self_signed', 'has_sni',
    'sni_entropy', 'flow_duration_sec', 'total_pkt_count',
    'consecutive_similar_iat_count', 'dominant_freq_power',
    'tls_sslv3', 'tls_1_0', 'tls_1_1'
]

fig, axes = plt.subplots(3, 5, figsize=(20, 10))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    if feat in df_all.columns:
        for label, color, alpha in [('benign', '#2ecc71', 0.5), ('malware', '#e74c3c', 0.5)]:
            subset = df_all[df_all['label'] == label][feat]
            axes[i].hist(subset, bins=30, alpha=alpha, color=color, label=label, density=True)
        axes[i].set_title(feat, fontsize=10)
        axes[i].legend(fontsize=8)

# Remove unused subplots
for j in range(len(key_features), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Feature Distributions: Benign vs Malware', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Data Preprocessing

Steps:
1. Encode labels (benign=0, malware=1)
2. Handle class imbalance with `scale_pos_weight`
3. Train/validation/test split (64/16/20)
4. Feature scaling (optional, XGBoost is tree-based but useful for reference)

In [ ]:
# === Encode labels ===
df_all['label_encoded'] = (df_all['label'].str.lower() != 'benign').astype(int)

# Verify label encoding
print("Label encoding:")
print(dict(df_all.groupby('label')['label_encoded'].first()))

# === Prepare feature matrix ===
# Use all features from the parser module
available_features = [f for f in ALL_FEATURES if f in df_all.columns]
missing_features = [f for f in ALL_FEATURES if f not in df_all.columns]
print(f"\nAvailable features: {len(available_features)}")
if missing_features:
    print(f"Missing features (will be zero-filled): {missing_features}")
    for f in missing_features:
        df_all[f] = 0.0

X = df_all[ALL_FEATURES].values.astype(np.float32)
y = df_all['label_encoded'].values

# Replace NaN/Inf
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\nFeature matrix: {X.shape}")
print(f"Labels: {y.shape}, malware ratio: {y.mean():.3f}")

In [ ]:
# === Train / Validation / Test split ===
# First: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Second: train vs val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SIZE/(1-TEST_SIZE),
    random_state=RANDOM_STATE, stratify=y_trainval
)

print(f"Train: {X_train.shape[0]} samples ({y_train.mean():.3f} malware)")
print(f"Val:   {X_val.shape[0]} samples ({y_val.mean():.3f} malware)")
print(f"Test:  {X_test.shape[0]} samples ({y_test.mean():.3f} malware)")

# === Compute class weights for imbalance handling ===
n_malware = y_train.sum()
n_benign = len(y_train) - n_malware
scale_pos_weight = n_benign / max(n_malware, 1)
print(f"\nscale_pos_weight: {scale_pos_weight:.2f}")

# Also compute sklearn class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(zip(np.unique(y_train), class_weights))
print(f"Class weights: {class_weight_dict}")

---
## 6. XGBoost Model Training

### Hyperparameters
- `scale_pos_weight` handles class imbalance
- `max_depth=6` prevents overfitting while capturing feature interactions
- `learning_rate=0.05` with `n_estimators=500` for stable convergence
- `subsample=0.8` / `colsample_bytree=0.8` for regularization

In [ ]:
# === XGBoost Classifier ===
model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    objective='binary:logistic',
    eval_metric=['auc', 'logloss'],
    random_state=RANDOM_STATE,
    n_jobs=-1,
    use_label_encoder=False,
)

print("Training XGBoost classifier...")
print(f"  scale_pos_weight = {scale_pos_weight:.2f}")
print(f"  n_estimators = {model.get_params()['n_estimators']}")
print(f"  max_depth = {model.get_params()['max_depth']}")
print(f"  learning_rate = {model.get_params()['learning_rate']}")

In [ ]:
# === Train with early stopping ===
eval_set = [(X_train, y_train), (X_val, y_val)]

model.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=50,
)

# Get training results
results = model.evals_result()
best_iteration = model.best_iteration
print(f"\nBest iteration: {best_iteration}")
print(f"Best validation AUC: {results['validation_1']['auc'][best_iteration]:.4f}")

In [ ]:
# === Training curves ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC
axes[0].plot(results['validation_0']['auc'], label='Train AUC', color='#3498db')
axes[0].plot(results['validation_1']['auc'], label='Val AUC', color='#e74c3c')
axes[0].axvline(x=best_iteration, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('ROC-AUC')
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('AUC')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log Loss
axes[1].plot(results['validation_0']['logloss'], label='Train Loss', color='#3498db')
axes[1].plot(results['validation_1']['logloss'], label='Val Loss', color='#e74c3c')
axes[1].axvline(x=best_iteration, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Log Loss')
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Curves', fontsize=14)
plt.tight_layout()
plt.show()

---
## 7. Threshold Tuning

The default 0.5 threshold is not optimal for imbalanced malware detection.
We tune using:
- **ROC-AUC curve** → find optimal threshold on the ROC curve
- **Precision-Recall curve** → maximize F1-score
- **Business rule**: minimize false negatives (malware missed)

In [ ]:
# === Get validation probabilities ===
y_val_proba = model.predict_proba(X_val)[:, 1]

# === ROC Curve ===
fpr, tpr, roc_thresholds = roc_curve(y_val, y_val_proba)
roc_auc = roc_auc_score(y_val, y_val_proba)

# Find optimal ROC threshold (Youden's J = TPR - FPR)
j_scores = tpr - fpr
roc_optimal_idx = np.argmax(j_scores)
roc_optimal_threshold = roc_thresholds[roc_optimal_idx]

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Optimal ROC threshold (Youden's J): {roc_optimal_threshold:.4f}")
print(f"  TPR={tpr[roc_optimal_idx]:.3f}, FPR={fpr[roc_optimal_idx]:.3f}")

# === Precision-Recall Curve ===
precision, recall, pr_thresholds = precision_recall_curve(y_val, y_val_proba)
avg_precision = average_precision_score(y_val, y_val_proba)

# Find threshold that maximizes F1
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
pr_optimal_idx = np.argmax(f1_scores)
pr_optimal_threshold = pr_thresholds[pr_optimal_idx]

print(f"\nAverage Precision (PR-AUC): {avg_precision:.4f}")
print(f"Optimal PR threshold (max F1): {pr_optimal_threshold:.4f}")
print(f"  Precision={precision[pr_optimal_idx]:.3f}, Recall={recall[pr_optimal_idx]:.3f}")
print(f"  F1={f1_scores[pr_optimal_idx]:.3f}")

In [ ]:
# === Plot ROC and PR curves ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#3498db', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', alpha=0.5)
axes[0].scatter(fpr[roc_optimal_idx], tpr[roc_optimal_idx], color='red', s=100, zorder=5,
                label=f'Optimal (t={roc_optimal_threshold:.3f})')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# PR Curve
axes[1].plot(recall, precision, color='#e74c3c', lw=2,
             label=f'PR (AP = {avg_precision:.3f})')
axes[1].scatter(recall[pr_optimal_idx], precision[pr_optimal_idx], color='blue', s=100, zorder=5,
                label=f'Max F1 (t={pr_optimal_threshold:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc='lower left')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Threshold Tuning', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === Select final threshold ===
# For malware detection, we prioritize high recall (catch all malware)
# while maintaining acceptable precision. Use the PR-optimal threshold
# unless recall is below 0.90, in which case lower the threshold.

# Evaluate at multiple thresholds
threshold_candidates = np.arange(0.2, 0.9, 0.05)
results_table = []

for t in threshold_candidates:
    y_pred = (y_val_proba >= t).astype(int)
    tp = ((y_pred == 1) & (y_val == 1)).sum()
    fp = ((y_pred == 1) & (y_val == 0)).sum()
    fn = ((y_pred == 0) & (y_val == 1)).sum()
    tn = ((y_pred == 0) & (y_val == 0)).sum()
    precision_val = tp / max(tp + fp, 1)
    recall_val = tp / max(tp + fn, 1)
    f1_val = 2 * precision_val * recall_val / max(precision_val + recall_val, 1e-9)
    results_table.append({
        'threshold': t,
        'precision': precision_val,
        'recall': recall_val,
        'f1': f1_val,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    })

df_thresh = pd.DataFrame(results_table)
print("Threshold Analysis:")
print(df_thresh.to_string(index=False))

# Select: max F1 threshold, but ensure recall >= 0.90
high_recall = df_thresh[df_thresh['recall'] >= 0.90]
if len(high_recall) > 0:
    final_threshold = high_recall.loc[high_recall['f1'].idxmax(), 'threshold']
else:
    final_threshold = df_thresh.loc[df_thresh['f1'].idxmax(), 'threshold']

print(f"\n✓ Selected threshold: {final_threshold:.2f}")

---
## 8. Test Set Evaluation

In [ ]:
# === Final evaluation on held-out test set ===
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

y_test_proba = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= final_threshold).astype(int)

# ─── Calculate all metrics ───
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
roc_auc = roc_auc_score(y_test, y_test_proba)
avg_precision = average_precision_score(y_test, y_test_proba)
cm = confusion_matrix(y_test, y_test_pred)

# ─── Print comprehensive evaluation ───
print("=" * 72)
print(" " * 20 + "FINAL TEST SET EVALUATION")
print("=" * 72)

# Metrics summary box
print("┌" + "─" * 44 + "┐")
print("│" + " PERFORMANCE METRICS".ljust(44) + "│")
print("├" + "─" * 44 + "┤")
print(f"│  Accuracy       : {accuracy * 100:>10.2f}%{' ' * 15}│")
print(f"│  Precision      : {precision:>10.4f}{' ' * 19}│")
print(f"│  Recall         : {recall:>10.4f}{' ' * 19}│")
print(f"│  F1-Score       : {f1:>10.4f}{' ' * 19}│")
print(f"│  ROC-AUC        : {roc_auc:>10.4f}{' ' * 19}│")
print(f"│  Avg Precision  : {avg_precision:>10.4f}{' ' * 19}│")
print("├" + "─" * 44 + "┤")
print(f"│  Threshold      : {final_threshold:>10.4f}{' ' * 19}│")
print(f"│  Test Samples   : {len(y_test):>10d}{' ' * 19}│")
print("└" + "─" * 44 + "┘")

# Classification report
print()
print("┌" + "─" * 56 + "┐")
print("│" + " CLASSIFICATION REPORT".ljust(56) + "│")
print("├" + "─" * 56 + "┤")
report_str = classification_report(
    y_test, y_test_pred,
    target_names=['Benign', 'Malware'],
    digits=4
)
for line in report_str.strip().split('\n'):
    print(f"│  {line.ljust(53)}│")
print("└" + "─" * 56 + "┘")

# Confusion matrix
tn, fp, fn, tp = cm.ravel()
print()
print("┌" + "─" * 56 + "┐")
print("│" + " CONFUSION MATRIX".ljust(56) + "│")
print("├" + "─" * 56 + "┤")
print(f"│{'':>20}│  PREDICTED  │{'':>20}│")
print(f"│{'':>20}│ Benign  Mal │{'':>20}│")
print("│" + "─" * 20 + "┼" + "─" * 12 + "┼" + "─" * 21 + "│")
print(f"│{'ACTUAL':>12} Benign │{:>6d}  {:>5d}  │{'':>20}│".format(tn, fp))
print(f"│{'ACTUAL':>12} Malware │{:>6d}  {:>5d}  │{'':>20}│".format(fn, tp))
print("│" + "─" * 20 + "┴" + "─" * 12 + "┴" + "─" * 21 + "│")
print(f"│  TP={tp}  FP={fp}  FN={fn}  TN={tn}".ljust(57) + "│")
print("└" + "─" * 56 + "┘")

print()
print("=" * 72)

# Confusion matrix heatmap
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malware'],
            yticklabels=['Benign', 'Malware'],
            ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

---
## 9. Feature Importance

In [ ]:
# === Feature importance (top 25) ===
importances = model.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': ALL_FEATURES,
    'importance': importances
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top_n = min(25, len(feat_imp_df))
feat_imp_df.head(top_n).plot(
    kind='barh', x='feature', y='importance', ax=ax,
    color='#3498db', legend=False
)
ax.set_xlabel('Importance (Gain)')
ax.set_title(f'Top {top_n} Feature Importances')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
for _, row in feat_imp_df.head(10).iterrows():
    print(f"  {row['feature']:35s} {row['importance']:.4f}")

---
## 10. Export Model Artifact

Save the model, threshold, feature list, scaler, and metrics as a single
`.pkl` artifact for the streaming inference pipeline.

In [ ]:
# === Cross-validation for confidence ===
print("Running 5-fold cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(
    model, X_trainval, y_trainval,
    cv=cv, scoring='roc_auc', n_jobs=-1
)
print(f"  CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Per-fold: {[f'{s:.4f}' for s in cv_scores]}")

In [ ]:
# === Build model artifact ===
model_artifact = {
    'model': model,
    'threshold': float(final_threshold),
    'features': list(ALL_FEATURES),
    'scaler': None,  # XGBoost doesn't need scaling
    'metrics': {
        'roc_auc_test': float(roc_auc_score(y_test, y_test_proba)),
        'avg_precision_test': float(average_precision_score(y_test, y_test_proba)),
        'cv_roc_auc_mean': float(cv_scores.mean()),
        'cv_roc_auc_std': float(cv_scores.std()),
        'n_features': len(ALL_FEATURES),
        'n_train': int(len(X_train)),
        'n_val': int(len(X_val)),
        'n_test': int(len(X_test)),
        'best_iteration': int(best_iteration),
        'scale_pos_weight': float(scale_pos_weight),
        'class_distribution': {
            'benign': int((y == 0).sum()),
            'malware': int((y == 1).sum()),
        },
    },
    'test_report': classification_report(
        y_test, y_test_pred,
        target_names=['benign', 'malware'],
        output_dict=True
    ),
}

# === Save artifact ===
joblib.dump(model_artifact, MODEL_ARTIFACT_PATH)
print(f"\n✓ Model artifact saved to: {MODEL_ARTIFACT_PATH}")
print(f"  File size: {os.path.getsize(MODEL_ARTIFACT_PATH) / 1024:.1f} KB")

# === Print summary ===
print("\n" + "=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Model: XGBoost Binary Classifier")
print(f"  Features: {len(ALL_FEATURES)}")
print(f"  Threshold: {final_threshold:.4f}")
print(f"  Test ROC-AUC: {model_artifact['metrics']['roc_auc_test']:.4f}")
print(f"  Test Avg Precision: {model_artifact['metrics']['avg_precision_test']:.4f}")
print(f"  CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Test F1 (Malware): {model_artifact['test_report']['malware']['f1-score']:.4f}")
print(f"  Test Recall (Malware): {model_artifact['test_report']['malware']['recall']:.4f}")
print(f"  Test Precision (Malware): {model_artifact['test_report']['malware']['precision']:.4f}")
print("=" * 60)
print(f"\nArtifact keys: {list(model_artifact.keys())}")

In [ ]:
# === Verify artifact loads correctly ===
loaded = joblib.load(MODEL_ARTIFACT_PATH)
print("Artifact loaded successfully.")
print(f"  Keys: {list(loaded.keys())}")
print(f"  Model type: {type(loaded['model']).__name__}")
print(f"  Threshold: {loaded['threshold']}")
print(f"  Feature count: {len(loaded['features'])}")
print(f"  Metrics: {list(loaded['metrics'].keys())}")

# Quick sanity check: predict on first test sample
test_sample = X_test[0:1]
proba = loaded['model'].predict_proba(test_sample)[0]
pred = 1 if proba[1] >= loaded['threshold'] else 0
print(f"\n  Sanity check: proba={proba}, pred={pred}, actual={y_test[0]}")

---
## 11. Integration Test

Verify the model works end-to-end with the detector and inference engine.

In [ ]:
# === Test detector integration ===
from ml_engine.malware_tls.detector import MalwareTLSDetector
from ml_engine.malware_tls.inference import MalwareTLSInferenceEngine

# Test detector
detector = MalwareTLSDetector(model_path=str(MODEL_ARTIFACT_PATH))
print(f"Detector loaded: {detector.metadata.name} v{detector.metadata.version}")
print(f"Threat class: {detector.threat_class.name}")
print(f"MITRE: {detector.threat_class.mitre_technique_id} - {detector.threat_class.mitre_technique_name}")

# Test prediction with a synthetic malware-like feature dict
test_features = {f: 0.0 for f in ALL_FEATURES}
test_features['iat_cv'] = 0.05
test_features['iat_regularity_score'] = 0.95
test_features['has_deprecated_cipher'] = 1.0
test_features['cert_self_signed'] = 1.0
test_features['cipher_suite_count'] = 2.0
test_features['has_sni'] = 0.0
test_features['tls_1_2'] = 1.0
test_features['consecutive_similar_iat_count'] = 25.0
test_features['dominant_freq_power'] = 5.0
test_features['proto_tcp'] = 1.0

test_context = {
    'src_ip': '192.168.1.100',
    'dst_ip': '10.0.0.50',
    'src_port': 52340,
    'dst_port': 443,
    'protocol': 'TCP',
}

prediction = detector.predict(test_features, test_context)
if prediction:
    print(f"\n⚠ Threat detected!")
    print(f"  Class: {prediction.threat_class}")
    print(f"  Confidence: {prediction.confidence}")
    print(f"  Severity: {prediction.severity}")
    print(f"  Z-score: {prediction.anomaly_zscore}")
else:
    print("\n✓ No threat detected (below threshold)")

# Test inference engine
engine = MalwareTLSInferenceEngine(model_path=str(MODEL_ARTIFACT_PATH))
engine.load()
proba = engine.predict_proba(test_features)
print(f"\nInference engine probability: {proba:.4f}")
print(f"Threshold: {engine.threshold}")
print(f"Is malware: {proba >= engine.threshold}")

---
## ✅ Training Complete

The model artifact is saved at:
```
ml_engine/malware_tls/malware_tls_streaming_model.pkl
```

### Artifact Contents
| Key | Description |
|---|---|
| `model` | Fitted XGBoost classifier |
| `threshold` | Optimal decision threshold |
| `features` | List of feature names (schema) |
| `scaler` | StandardScaler (None for XGBoost) |
| `metrics` | Training/validation/test metrics |
| `test_report` | Full classification report |

### Usage
```python
from ml_engine.malware_tls.detector import MalwareTLSDetector
detector = MalwareTLSDetector()  # Auto-loads default model path
prediction = detector.predict(features, context)
```

### Next Steps
1. Replace synthetic data with real CIC-MalAnal2020 + Brad Duncan datasets
2. Re-run this notebook to train on real data
3. Run `inference.py` on test PCAPs to validate end-to-end